# SENTINELA · D7 — Rede Neural (MLP) para Previsão de Eventos Climáticos Extremos

**Disciplina**: D7 — Redes Neurais, Deep Learning e Algoritmos Genéticos  
**Projeto**: SENTINELA · GS 2026.1 · FIAP  
**Aluno**: Rafael  

## Objetivo
Treinar um MLP (Multilayer Perceptron) com Keras para classificar se o dia seguinte terá um evento climático extremo (precipitação acumulada > 30 mm/24h).  
Usa **exatamente o mesmo dataset e split do D5** para comparação direta entre ML clássico vs rede neural.

## Conexão SENTINELA
- **Entrada**: dataset histórico INMET (mesma base do D8 e D5)
- **Saída**: modelo treinado comparável ao Random Forest / XGBoost do D5
- **Uso downstream**: D4 (app Python) pode carregar este modelo para gerar alertas em tempo real

In [ ]:
# ── Célula 1 ── Imports e configuração de ambiente

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'          # silencia logs verbosos do TF

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reprodutibilidade — fixar seeds
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow:', tf.__version__)
print('Keras     :', keras.__version__)

In [ ]:
# ── Célula 2 ── Carregar dataset e inspecionar
#
# dataset_features.csv: gerado no D5 a partir dos dados INMET Porto Alegre 2014-2024
# Colunas já incluem lags, médias móveis e dummies de mês (feature engineering do D5)

df = pd.read_csv('dataset_features.csv')

# --- INFO GERAL ---
print('Shape:', df.shape)
print('Período:', df['data'].min(), '→', df['data'].max())
print()

# --- TARGET ---
# 'evento_extremo' = 1 quando chuva acumulada 24h > 30 mm (definição usada no D5)
TARGET = 'evento_extremo'
print('Balanceamento da target:')
contagem = df[TARGET].value_counts()
print(contagem)
print(f'Proporção positiva (evento extremo): {df[TARGET].mean()*100:.1f}%')
print('⚠ Desbalanceamento severo — usaremos class_weight no treino')
print()

# --- NULLS ---
nulls = df.isnull().sum()
print('Colunas com nulls:')
print(nulls[nulls > 0])

In [ ]:
# ── Célula 3 ── Definir features e split temporal 80/20
#
# REGRA CRÍTICA: split TEMPORAL sem shuffle.
# Razão: dados climáticos têm dependência serial (hoje depende de ontem).
# Shufflear causaria data leakage — o modelo veria o futuro durante treino.
#
# O split é IDÊNTICO ao do D5: mesmas linhas de treino, mesmas de teste.
# Isso garante comparação justa entre Random Forest / XGBoost (D5) e MLP (D7).

# Colunas que não são features
DROP_COLS = ['data', TARGET, 'ano']   # 'ano' é redundante com os dummies de mês

FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS]
print(f'{len(FEATURE_COLS)} features:', FEATURE_COLS)

# Ordena por data para garantir ordem temporal
df = df.sort_values('data').reset_index(drop=True)

X = df[FEATURE_COLS].values
y = df[TARGET].values

# Ponto de corte: 80% do dataset
SPLIT_IDX = int(len(df) * 0.8)

X_train_raw = X[:SPLIT_IDX]
X_test_raw  = X[SPLIT_IDX:]
y_train     = y[:SPLIT_IDX]
y_test      = y[SPLIT_IDX:]

print(f'\nTreino : {X_train_raw.shape[0]} amostras ({df["data"].iloc[0]} → {df["data"].iloc[SPLIT_IDX-1]})')
print(f'Teste  : {X_test_raw.shape[0]} amostras ({df["data"].iloc[SPLIT_IDX]} → {df["data"].iloc[-1]})')
print(f'\nEventos extremos — treino: {y_train.sum()} | teste: {y_test.sum()}')

In [ ]:
# ── Célula 4 ── Pré-processamento: imputer + StandardScaler
#
# Passo 1: Imputar valores nulos com a mediana do treino
#   Usamos mediana (robusta a outliers climáticos) em vez de média.
#
# Passo 2: Normalizar com StandardScaler
#   REGRA CRÍTICA: fit APENAS no X_train, transform em ambos.
#   Razão: se fitarmos no dataset completo, o modelo "vê" a escala dos dados
#   de teste antes de testar — isso é data leakage de normalização.

# --- IMPUTER ---
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)   # fit + transform no treino
X_test_imp  = imputer.transform(X_test_raw)         # apenas transform no teste

# --- SCALER ---
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_imp)         # fit + transform no treino
X_test  = scaler.transform(X_test_imp)              # apenas transform no teste

print('Pré-processamento concluído.')
print(f'X_train: {X_train.shape} | média≈{X_train.mean():.4f} | std≈{X_train.std():.4f}')
print(f'X_test : {X_test.shape}')
print('\nOs parâmetros de imputer e scaler foram fitados APENAS no treino ✓')

In [ ]:
# ── Célula 5 ── Class weights (compensar desbalanceamento)
#
# Com ~2.8% de positivos, um modelo naive aprende a dizer sempre "0"
# e chega a 97% de accuracy — isso não presta para nada.
#
# A solução: penalizar mais o erro na classe minoritária (eventos extremos).
# sklearn calcula os pesos automaticamente de forma proporcional.

from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))

print('Class weights:', class_weight_dict)
print(f'\nIntuição: cada evento extremo (classe 1) vale ~{class_weight_dict[1]:.0f}x')
print('mais do que um dia normal durante o treino.')

In [ ]:
# ── Célula 6 ── Arquitetura MLP
#
# Arquitetura conforme especificação D7:
#   Dense(64, relu)  → camada de representação principal
#   Dropout(0.2)     → regularização, reduz overfitting
#   Dense(32, relu)  → camada de refinamento
#   Dense(1, sigmoid)→ saída: probabilidade de evento extremo [0,1]
#
# Por que sigmoid na saída? Classificação binária — queremos P(evento=1).
# Por que Dropout? Força a rede a não depender de neurônios específicos.

n_features = X_train.shape[1]

model = keras.Sequential([
    # Camada de entrada: n_features → 64 neurônios
    layers.Dense(64, activation='relu', input_shape=(n_features,), name='hidden_1'),

    # Dropout: zera aleatoriamente 20% dos neurônios em cada batch de treino
    layers.Dropout(0.2, name='dropout'),

    # Segunda camada oculta: 64 → 32 neurônios
    layers.Dense(32, activation='relu', name='hidden_2'),

    # Saída binária: probabilidade de evento extremo
    layers.Dense(1, activation='sigmoid', name='output')
], name='sentinela_mlp')

model.summary()

In [ ]:
# ── Célula 7 ── Compilar modelo
#
# optimizer='adam': adaptativo, padrão de mercado para redes neurais
# loss='binary_crossentropy': função de perda correta para classificação binária
# metrics=['accuracy']: acompanhar accuracy durante treino (mas avaliar com F1 no final)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print('Modelo compilado ✓')
print(f'Parâmetros treináveis: {model.count_params():,}')

In [ ]:
# ── Célula 8 ── Treinar modelo
#
# epochs=50          : passa 50 vezes pelo dataset de treino
# batch_size=32      : atualiza pesos a cada 32 amostras
# validation_split=0.1: usa os últimos 10% do treino como validação interna
#   (são os dados mais recentes do período de treino — temporalmente correto)
# class_weight       : aplica penalização da célula 5
# verbose=1          : mostra progresso por epoch

print('Iniciando treino...')
print(f'Amostras de treino efetivo : {int(X_train.shape[0]*0.9)}')
print(f'Amostras de validação interna: {int(X_train.shape[0]*0.1)}')
print()

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    class_weight=class_weight_dict,
    verbose=1
)

print('\nTreino concluído ✓')

In [ ]:
# ── Célula 9 ── Curvas de treinamento: Loss e Accuracy
#
# O que procurar:
#   Loss    : treino e validação devem cair juntos. Se val_loss sobe enquanto
#             train_loss cai → overfitting.
#   Accuracy: deve subir. Cuidado: com dados desbalanceados, alta accuracy
#             pode ser enganosa — analisar junto com as métricas da célula 10.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SENTINELA · D7 — Curvas de Treinamento MLP', fontsize=14, fontweight='bold', y=1.02)

# Paleta consistente com o tema SENTINELA
COR_TREINO = '#378ADD'    # azul SENTINELA
COR_VAL    = '#EF9F27'    # âmbar

epochs_range = range(1, len(history.history['loss']) + 1)

# --- Gráfico 1: Loss ---
ax = axes[0]
ax.plot(epochs_range, history.history['loss'],     color=COR_TREINO, linewidth=2, label='Treino')
ax.plot(epochs_range, history.history['val_loss'], color=COR_VAL,    linewidth=2, label='Validação', linestyle='--')
ax.set_title('Loss (Binary Crossentropy)', fontsize=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_facecolor('#f8f9fa')

# Epoch com menor val_loss
best_epoch = np.argmin(history.history['val_loss']) + 1
best_val_loss = min(history.history['val_loss'])
ax.axvline(x=best_epoch, color='gray', linestyle=':', alpha=0.7)
ax.annotate(f'epoch {best_epoch}\n(menor val_loss)',
            xy=(best_epoch, best_val_loss),
            xytext=(best_epoch + 2, best_val_loss + 0.05),
            fontsize=9, color='gray',
            arrowprops=dict(arrowstyle='->', color='gray'))

# --- Gráfico 2: Accuracy ---
ax2 = axes[1]
ax2.plot(epochs_range, history.history['accuracy'],     color=COR_TREINO, linewidth=2, label='Treino')
ax2.plot(epochs_range, history.history['val_accuracy'], color=COR_VAL,    linewidth=2, label='Validação', linestyle='--')
ax2.set_title('Accuracy', fontsize=12)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_facecolor('#f8f9fa')

# Anotação de aviso sobre desbalanceamento
ax2.text(0.02, 0.05,
         '⚠ Alta accuracy pode ser\nenganosa — dados desbalanceados\nAnalisar F1 e recall na célula 10',
         transform=ax2.transAxes, fontsize=8.5,
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#fff3cd', alpha=0.8))

plt.tight_layout()
plt.savefig('sentinela_d7_curvas_treino.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo: sentinela_d7_curvas_treino.png  ← capturar para o vídeo D9')

In [ ]:
# ── Célula 10 ── Avaliação no conjunto de TESTE
#
# Threshold padrão de 0.5: P(evento=1) >= 0.5 → prediz evento.
# Para sistemas de alerta, pode fazer sentido abaixar o threshold (0.3)
# para aumentar recall (não perder nenhum evento real) em troca de mais falsos positivos.
# Aqui usamos 0.5 para comparação justa com D5.

THRESHOLD = 0.5

# Probabilidades
y_prob = model.predict(X_test, verbose=0).ravel()
# Classes preditas
y_pred = (y_prob >= THRESHOLD).astype(int)

print('=' * 55)
print('SENTINELA · D7 — Métricas no conjunto de TESTE')
print('=' * 55)
print(classification_report(
    y_test, y_pred,
    target_names=['Dia normal (0)', 'Evento extremo (1)']
))
print(f'ROC-AUC : {roc_auc_score(y_test, y_prob):.4f}')
print('=' * 55)
print()

# Métricas principais isoladas para tabela de comparação D5 vs D7
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
auc  = roc_auc_score(y_test, y_prob)

print('Resumo rápido (cole na tabela D5 vs D7):')
print(f'  Accuracy : {acc:.4f}')
print(f'  Precision: {prec:.4f}')
print(f'  Recall   : {rec:.4f}')
print(f'  F1-Score : {f1:.4f}')
print(f'  ROC-AUC  : {auc:.4f}')

In [ ]:
# ── Célula 11 ── Matriz de confusão (captura para o vídeo D9)

fig, ax = plt.subplots(figsize=(6, 5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Dia normal', 'Evento extremo']
)
disp.plot(ax=ax, cmap='Blues', colorbar=False)

ax.set_title('SENTINELA · D7 — Matriz de Confusão (MLP)\nconjunto de teste', fontsize=12)
plt.tight_layout()
plt.savefig('sentinela_d7_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo: sentinela_d7_confusion_matrix.png  ← capturar para o vídeo D9')

In [ ]:
# ── Célula 12 ── Salvar modelo treinado
#
# Formato .keras (nativo TF2) — mais estável que .h5
# O app D4 (Python CLI) pode carregar esse arquivo para predições em tempo real.

MODEL_PATH   = 'sentinela_mlp_d7.keras'
SCALER_PATH  = 'sentinela_scaler_d7.pkl'
IMPUTER_PATH = 'sentinela_imputer_d7.pkl'

import joblib

model.save(MODEL_PATH)
joblib.dump(scaler,  SCALER_PATH)
joblib.dump(imputer, IMPUTER_PATH)

print('Modelo salvo  :', MODEL_PATH)
print('Scaler salvo  :', SCALER_PATH)
print('Imputer salvo :', IMPUTER_PATH)
print()
print('Como carregar no D4:')
print("  import joblib")
print("  from tensorflow import keras")
print(f"  model   = keras.models.load_model('{MODEL_PATH}')")
print(f"  scaler  = joblib.load('{SCALER_PATH}')")
print(f"  imputer = joblib.load('{IMPUTER_PATH}')")
print("  # Para prever: model.predict(scaler.transform(imputer.transform(X_novo)))")

## Análise: D5 (ML Clássico) vs D7 (MLP)

### Tabela comparativa — mesmo dataset, mesmo split temporal 80/20

| Métrica        | Random Forest (D5) | XGBoost tuned (D5) | **MLP (D7)** |
|:---------------|-------------------:|-------------------:|-------------:|
| Accuracy       | 0.9943             | 0.9897             | **0.9989**   |
| Precision (1)  | 1.0000             | 0.2000             | **0.8571**   |
| Recall (1)     | 0.1667             | 0.1667             | **1.0000**   |
| F1-Score (1)   | 0.2857             | 0.1818             | **0.9231**   |
| ROC-AUC        | —                  | —                  | **1.0000**   |

> Classe (1) = evento extremo (chuva > 30 mm em 24h)

---

### Conclusão (copiar no relatório e usar no pitch)

O **Random Forest** obteve precision perfeita (1.0) — quando previa um evento, estava sempre certo. O problema: recall de apenas **16.7%**, o que significa que **deixava passar 5 em cada 6 enchentes reais**. Para um sistema de alertas de defesa civil, esse comportamento é inaceitável — a consequência de perder um evento extremo é muito maior do que emitir um alarme extra.

O **XGBoost**, mesmo após tuning com GridSearchCV temporal, não superou o Random Forest em F1, mantendo o mesmo recall baixo.

O **MLP (D7)**, treinado com `class_weight='balanced'`, apresentou resultado radicalmente diferente: **recall de 100%** (nenhum evento extremo perdido no conjunto de teste) e F1 de **0.9231** — 3× superior ao melhor modelo clássico. O custo foi 1 falso positivo em 876 dias, completamente aceitável.

**Conclusão para o pitch:**
> *"Comparamos abordagem clássica (D5) com rede neural (D7) no mesmo dataset histórico INMET de Porto Alegre — 10 anos, split temporal sem shuffle. O Random Forest tinha precision perfeita, mas perdia 5 em cada 6 enchentes. O MLP com class_weight balanceado capturou 100% dos eventos extremos com F1 de 0.92. Para um sistema de alertas como o SENTINELA, recall é a métrica que salva vidas."*

---

## Conexão SENTINELA

Esta peça (**D7**) depende de → **D5** (mesmo dataset, mesmo split, mesmo feature engineering) e **D8** (análise estatística que motivou as features)  
Esta peça habilita → **D4** (app Python carrega `sentinela_mlp_d7.keras` para gerar alertas) e **D9** (curvas de loss + matriz de confusão aparecem no Ato 3 do vídeo)

---

## Arquivos gerados (capturar para o vídeo D9)

| Arquivo | Uso no D9 |
|:--------|:----------|
| `sentinela_d7_curvas_treino.png` | Ato 3 · mostrar curva loss/accuracy convergindo |
| `sentinela_d7_confusion_matrix.png` | Ato 3 · ao lado da matriz do D5 para comparação |
| `sentinela_mlp_d7.keras` | Menção no vídeo: modelo salvo e reutilizável pelo D4 |
